In [137]:
# Assignment 2 - LLM Document Analysis
# Mauricio F. M. Costa (ma038677@ucf.edu)
# 4855025

# Installing essential libraries

In [138]:
%pip install requests pandas beautifulsoup4 lxml

In [139]:
import re
import json
import time
import requests
import subprocess
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files
from urllib.parse import urljoin

# Let's retrieve company and filing metadata

In [140]:
HEADERS = {
  "User-Agent": "LLM Document Analysis by ma038677@ucf.edu",
  "Accept-Encoding": "gzip, deflate"
}

session = requests.Session()
session.headers.update(HEADERS)

def get_sec_json(url):
  response = session.get(url, timeout=30)
  response.raise_for_status()
  time.sleep(0.2)
  return response.json()

# Getting company CIK from ticker symbol
def get_company_cik(ticker):
  url = "https://www.sec.gov/files/company_tickers.json"
  companies = get_sec_json(url)

  for company in companies.values():
    if company["ticker"].upper() == ticker.upper():
      return str(company["cik_str"]).zfill(10)

  raise ValueError(f"Ticker {ticker} not found")


# Retrieving recent Form 8-K filings
def get_8k_filings(ticker, limit=10):
  cik = get_company_cik(ticker)

  url = f"https://data.sec.gov/submissions/CIK{cik}.json"
  data = get_sec_json(url)

  company_name = data["name"]
  recent = data["filings"]["recent"]

  filings = []

  for i, form in enumerate(recent["form"]):
    if form != "8-K":
      continue

    accession = recent["accessionNumber"][i]
    document = recent["primaryDocument"][i]

    filing_url = (
      f"https://www.sec.gov/Archives/edgar/data/"
      f"{int(cik)}/{accession.replace('-', '')}/{document}"
    )

    filings.append({
      "company_name": company_name,
      "stock_name": ticker.upper(),
      "filing_time": recent["acceptanceDateTime"][i],
      "filing_date": recent["filingDate"][i],
      "filing_url": filing_url
    })

    if len(filings) >= limit:
      break

  return pd.DataFrame(filings)


# Testing with NVIDIA for example
df = get_8k_filings("NVDA", limit=10)

print(f"Retrieved {len(df)} filings")
display(df)


Retrieved 10 filings


,company_name,stock_name,filing_time,filing_date,filing_url
0,NVIDIA CORP,NVDA,2026-09-03T12:03:56.000Z,2026-09-03,https://www.sec.gov/Archives/edgar/data/104581...
1,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,2026-08-26,https://www.sec.gov/Archives/edgar/data/104581...
2,NVIDIA CORP,NVDA,2026-08-17T12:41:33.000Z,2026-08-17,https://www.sec.gov/Archives/edgar/data/104581...
3,NVIDIA CORP,NVDA,2026-07-02T13:23:16.000Z,2026-07-02,https://www.sec.gov/Archives/edgar/data/104581...
4,NVIDIA CORP,NVDA,2026-06-30T20:27:26.000Z,2026-06-30,https://www.sec.gov/Archives/edgar/data/104581...
5,NVIDIA CORP,NVDA,2026-06-18T20:00:24.000Z,2026-06-18,https://www.sec.gov/Archives/edgar/data/104581...
6,NVIDIA CORP,NVDA,2026-05-20T20:21:19.000Z,2026-05-20,https://www.sec.gov/Archives/edgar/data/104581...
7,NVIDIA CORP,NVDA,2026-05-08T20:35:07.000Z,2026-05-08,https://www.sec.gov/Archives/edgar/data/104581...
8,NVIDIA CORP,NVDA,2026-04-27T20:10:30.000Z,2026-04-27,https://www.sec.gov/Archives/edgar/data/104581...
9,NVIDIA CORP,NVDA,2026-03-06T21:11:25.000Z,2026-03-06,https://www.sec.gov/Archives/edgar/data/104581...


# Continuing to retrieve contents from the SEC, parsing their HTML into readable text and watching out for exhibits that contain announcements of new products.

In [141]:
# Converting SEC HTML documents into readable text
def clean_html(html):
  soup = BeautifulSoup(html, "lxml")

  # Removing elements that don't contain useful content
  for tag in soup(["script", "style", "noscript"]):
    tag.decompose()

  text = soup.get_text(separator=" ", strip=True)

  # Removing too much whitespace
  text = re.sub(r"\s+", " ", text)

  return text.strip()


# Helper to download 1 SEC document
def download_document(url):
  response = session.get(url, timeout=30)
  response.raise_for_status()
  time.sleep(0.2)

  return clean_html(response.content)


# Finding Exhibit 99 documents associated with an 8-K
def find_exhibits(filing_url):
  base_url = filing_url.rsplit("/", 1)[0]

  accession = base_url.split("/")[-1]
  accession_dashed = (
    f"{accession[:10]}-{accession[10:12]}-{accession[12:]}"
  )

  index_url = f"{base_url}/{accession_dashed}-index.htm"

  response = session.get(index_url, timeout=30)
  response.raise_for_status()
  time.sleep(0.2)

  soup = BeautifulSoup(response.content, "lxml")
  exhibits = []

  for table in soup.find_all("table", class_="tableFile"):
    for tr in table.find_all("tr"):
      cells = tr.find_all("td")

      if len(cells) < 4:
        continue

      document_type = cells[3].get_text(strip=True)

      if not document_type.startswith("EX-99"):
        continue

      link = cells[2].find("a", href=True)

      if link:
        exhibit_url = urljoin(
          index_url, link["href"]
        )

        # Handling only HTML and text exhibits for now
        if exhibit_url.lower().split("?")[0].endswith(
          (".htm", ".html", ".txt", ".xhtml")
        ):exhibits.append({
          "document_type": document_type,
          "source_url": exhibit_url
        })

  return exhibits


# Downloading the main filing and its exhibits
def process_filing(filing):
  documents = []

  sources = [{
    "document_type": "8-K",
    "source_url": filing["filing_url"]
  }]

  sources.extend(find_exhibits(filing["filing_url"]))

  for source in sources:
    try:
      text = download_document(source["source_url"])

      documents.append({
        "company_name": filing["company_name"],
        "stock_name": filing["stock_name"],
        "filing_time": filing["filing_time"],
        "document_type": source["document_type"],
        "source_url": source["source_url"],
        "text": text,
        "characters": len(text)
      })

    except Exception as e:
      print(f"Could not download {source['source_url']}")
      print(e)

  return documents


# Testing (again) with the first 2 NVIDIA filings now
documents = []

for _, filing in df.head(2).iterrows():
  documents.extend(process_filing(filing))

documents_df = pd.DataFrame(documents)

print(f"Downloaded {len(documents_df)} documents")

display(documents_df[
  ["company_name", "document_type", "characters", "source_url"]
])

# Preview the beginning of the first document
if not documents_df.empty:
  print("\nDOCUMENT TEXT PREVIEW:\n")
  print(documents_df.iloc[0]["text"][:1500])


/tmp/ipykernel_702/2962302928.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Downloaded 4 documents


,company_name,document_type,characters,source_url
0,NVIDIA CORP,8-K,8074,https://www.sec.gov/Archives/edgar/data/104581...
1,NVIDIA CORP,8-K,3988,https://www.sec.gov/Archives/edgar/data/104581...
2,NVIDIA CORP,EX-99.1,22347,https://www.sec.gov/Archives/edgar/data/104581...
3,NVIDIA CORP,EX-99.2,19722,https://www.sec.gov/Archives/edgar/data/104581...



DOCUMENT TEXT PREVIEW:

nvda-20260902 0001045810 false 0001045810 2026-09-02 2026-09-02 UNITED STATES SECURITIES AND EXCHANGE COMMISSION WASHINGTON, DC 20549 ______________ FORM 8-K CURRENT REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 Date of Report (Date of earliest event reported): September 2, 2026 NVIDIA CORP ORATION (Exact name of registrant as specified in its charter) Delaware 0-23985 94-3177549 (State or other jurisdiction (Commission (IRS Employer of incorporation) File Number) Identification No.) 2788 San Tomas Expressway , Santa Clara , CA 95051 (Address of principal executive offices) (Zip Code) Registrant’s telephone number, including area code: ( 408 ) 486-2000 Not Applicable (Former name or former address, if changed since last report) Check the appropriate box below if the Form 8-K filing is intended to simultaneously satisfy the filing obligation of the registrant under any of the following provisions: ☐ Written communications pursuant

# Saving SEC docs locally so I don't run into any rate limits from the SEC.

In [142]:
# Save the documents for Google Colab
documents_df.to_json(
  "sec_8k_documents.json",
  orient="records",
  force_ascii=False,
  indent=2
)

print("SEC Documents saved successfully!")

SEC Documents saved successfully!


# Setting up "local" LLM here on google colab

In [143]:
!nvidia-smi

Tue Nov  3 04:47:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |    2579MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [144]:
# I was having issues with the zstd package when installing Ollama, so:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [145]:
# Installing Ollama:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [146]:
# Installing Ollama server
OLLAMA_URL = "http://localhost:11434"

try:
  requests.get(OLLAMA_URL, timeout=2).raise_for_status()
  print("Ollama server is already running!")
except requests.RequestException:
  log_file = open("/tmp/ollama.log", "w")

  ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
  )

  for _ in range(30):
    try:
      requests.get(OLLAMA_URL, timeout=2).raise_for_status()
      print("Ollama server started successfully!")
      break
    except requests.RequestException:
      time.sleep(1)
  else:
    raise RuntimeError("Ollama failed to start. Check /tmp/ollama.log")


Ollama server is already running!


In [147]:
# Downloading model hre to Google Colab
!ollama pull llama3.2:3b

In [148]:
# Querying LLM
response = requests.post(
  "http://localhost:11434/api/chat",
  json={
    "model": "llama3.2:3b",
    "messages": [
      {
        "role": "user",
        "content": (
          "Explain what SEC Form 8-K is in one sentence."
        )
      }
    ],
    "stream": False,
    "options": {
      "temperature": 0
    }
  },
  timeout=180
)

response.raise_for_status()

answer = response.json()["message"]["content"]

print("=== LLM Response: ===")
print(answer)


=== LLM Response: ===
SEC Form 8-K is a quarterly and annual disclosure form filed with the U.S. Securities and Exchange Commission (SEC) by publicly traded companies to report material events, such as mergers and acquisitions, changes in corporate governance, or significant financial transactions.


# Product extraction helper function

In [149]:
MODEL = "llama3.2:3b"
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"

# Expected output structure (schema):
PRODUCT_SCHEMA = {
  "type": "object",
  "properties": {
    "products": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "new_product": {"type": "string"},
          "product_description": {"type": "string"},
          "evidence": {"type": "string"}
        },
        "required": [
          "new_product",
          "product_description",
          "evidence"
        ]
      }
    }
  },
  "required": ["products"]
}


def extract_products(text):
  prompt = f"""
You are a financial research analyst specializing in
Named Entity Recognition (NER) and SEC filings.

TASK:
Extract NEW product announcements from the document.

DEFINITION:
A new product announcement must explicitly introduce, launch, unveil, or debut a specific product.

INCLUDE:
- Newly launched products
- Newly unveiled technologies or platforms
- Newly introduced commercial products
- Multiple distinct product launches

EXCLUDE:
- Existing products mentioned in passing
- Products only reporting production milestones
- Financial results or benchmark achievements
- Partnerships or customer adoption announcements
- General corporate updates
- Product improvements without a distinct new launch
- Products whose launch status is uncertain

EXTRACTION RULES:
1. Extract the complete official product name.
2. Identify the product as a PRODUCT entity.
3. Provide a factual description under 180 characters.
4. Copy an exact supporting quote from the document.
5. Do not invent or paraphrase evidence!
6. Do not infer a launch from a product mention alone.
7. If there are no qualifying products, return an empty products list.
8. Return valid JSON matching the specified schema.

DOCUMENT:
{text}
"""
  response = requests.post(
    OLLAMA_CHAT_URL,
    json={
      "model": MODEL,
      "messages": [
        {
          "role": "user",
          "content": prompt
        }
      ],
      "format": PRODUCT_SCHEMA,
      "stream": False,
      "options": {
        "temperature": 0,
        "num_ctx": 4096
      }
    },
    timeout=240
  )

  response.raise_for_status()

  result = json.loads(
    response.json()["message"]["content"]
  )

  return result["products"]


In [150]:
# Testing on one exhibit only for now:

# Selecting first Exhibit 99
exhibits = documents_df[
  documents_df["document_type"].str.startswith("EX-99")
]

test_document = exhibits.iloc[0]

print("Company:", test_document["company_name"])
print("Document:", test_document["document_type"])
print("Characters:", test_document["characters"])

# Starting with a small section
sample_text = test_document["text"][:6000]

print("\nAnalyzing with Llama 3.2...\n")

products = extract_products(sample_text)

if products:
  for product in products:
    print("PRODUCT:", product["new_product"])
    print("DESCRIPTION:", product["product_description"])
    print("EVIDENCE:", product["evidence"])
    print("-" * 50)
else:
  print("No new product announcements identified.")


Company: NVIDIA CORP
Document: EX-99.1
Characters: 22347

Analyzing with Llama 3.2...

PRODUCT: NVIDIA Vera Rubin
DESCRIPTION: Platform for Data Center compute
EVIDENCE: Announced the NVIDIA Vera Rubin platform is ramping into full production with racks running at partners including CoreWeave, Google Cloud, Microsoft Azure, Oracle Cloud Infrastructure and Nebius.
--------------------------------------------------
PRODUCT: NVIDIA Spectrum-6
DESCRIPTION: Switch systems for Data Center
EVIDENCE: Revealed that NVIDIA Spectrum-6 switch systems — supporting both pluggable and co-packaged optics as part of the NVIDIA Vera Rubin platform — are arriving across the world’s gigascale AI factories.
--------------------------------------------------
PRODUCT: NVIDIA Vera
DESCRIPTION: First CPU built for AI agents
EVIDENCE: Unveiled NVIDIA Vera, the first CPU built for AI agents, with broad adoption planned across the world’s leading technology providers.
---------------------------------------------

# Step 5: Processing documents and generating the actual CSV...

In [151]:
# Splitting long documents into chunks
def chunk_text(text, chunk_size=5000, overlap=500):
  chunks = []
  start = 0

  while start < len(text):
    end = min(start + chunk_size, len(text))
    chunks.append(text[start:end])

    if end == len(text):
      break

    start = end - overlap

  return chunks


# Analyzing all downloaded docs
records = []

for i, document in documents_df.iterrows():
  print(
    f"Processing document {i + 1}/{len(documents_df)}: "
    f"{document['company_name']} - "
    f"{document['document_type']}"
  )

  chunks = chunk_text(document["text"])

  for chunk in chunks:
    products = extract_products(chunk)

    for product in products:
      name = product.get("new_product", "").strip()
      description = product.get(
        "product_description", ""
      ).strip()

      if not name or not description:
        continue

      # Ensuring description is under 180 characters
      description = description[:179]

      records.append({
        "company_name": document["company_name"],
        "stock_name": document["stock_name"],
        "filing_time": document["filing_time"],
        "new_product": name,
        "product_description": description
      })


# Creating the required CSV structure
columns = [
  "company_name",
  "stock_name",
  "filing_time",
  "new_product",
  "product_description"
]

results_df = pd.DataFrame(records, columns=columns)

# Let's now remove duplicate product names within each filing
results_df = results_df.drop_duplicates(
  subset=[
    "stock_name",
    "filing_time",
    "new_product"
  ]
)

# Savng results
results_df.to_csv(
  "sec_8k_products.csv",
  index=False,
  encoding="utf-8"
)

print("\nExtraction is now completed!")
print(f"Products extracted: {len(results_df)}")

display(results_df)

Processing document 1/4: NVIDIA CORP - 8-K
Processing document 2/4: NVIDIA CORP - 8-K
Processing document 3/4: NVIDIA CORP - EX-99.1
Processing document 4/4: NVIDIA CORP - EX-99.2

Extraction is now completed!
Products extracted: 34


,company_name,stock_name,filing_time,new_product,product_description
0,NVIDIA CORP,NVDA,2026-09-03T12:03:56.000Z,Hugging Face,"Platform and community for developing, sharing..."
1,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Announces Financial Results for Second ...,Financial results for Second Quarter Fiscal 2027
2,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera Rubin platform,Ramping into full production
3,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Spectrum™-6 switch systems,Arriving across the world’s gigascale AI facto...
4,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera,First CPU built for AI agents
5,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Groq 3 LPX,Now in full production
6,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera BlueField™-4 STX,New security innovations
7,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA DSX,Providing infrastructure builders a complete p...
8,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Agen,"New software, open source models and partnerships"
9,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Blackwell,Industry-leading AI infrastructure benchmark


# Downloading CSV:

In [152]:
# files.download("Assignment_2_SEC_8k_products_MCOSTA.csv")

# Retrieving additional SEC filings (to get to the 100 min)

In [153]:
# Companies to analyze (s&p):
tickers = ["NVDA", "AMD", "MSFT", "AAPL", "META"]

# Keeping all previously downloaded documents
all_documents = documents_df.to_dict("records")

existing_urls = {
  doc["source_url"] for doc in all_documents
}

for ticker in tickers:
  # Let's skip companies that were already analyzed
  if ticker in documents_df["stock_name"].values:
      print(f"{ticker}: Already collected")
      continue

  print(f"\nRetrieving filings for {ticker}...")

  try:
      # Retrieving 2 filings per missing company
      filings = get_8k_filings(ticker, limit=2)

      for _, filing in filings.iterrows():

        try:
          downloaded = process_filing(filing)

          for document in downloaded:
            url = document["source_url"]

            if url not in existing_urls:
              all_documents.append(document)
              existing_urls.add(url)

        except Exception as e:
          print(f"Error processing filing: {e}")

  except Exception as e:
    print(f"Error retrieving {ticker}: {e}")


# Updating df
documents_df = pd.DataFrame(all_documents)

print("\nDATA COLLECTION COMPLETED")
print(f"Total documents: {len(documents_df)}")
print(f"Companies: {documents_df['stock_name'].nunique()}")

display(
    documents_df.groupby("stock_name")
    .size()
    .reset_index(name="documents")
)

# Save backup
documents_df.to_json(
    "sec_8k_documents_expanded.json",
    orient="records",
    force_ascii=False
)


NVDA: Already collected

Retrieving filings for AMD...


/tmp/ipykernel_702/2962302928.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")



Retrieving filings for MSFT...

Retrieving filings for AAPL...

Retrieving filings for META...

DATA COLLECTION COMPLETED
Total documents: 17
Companies: 5


,stock_name,documents
0,AAPL,4
1,AMD,3
2,META,3
3,MSFT,3
4,NVDA,4


In [154]:
# Running the extraction again:

# Splitting long documents into chunks
def chunk_text(text, chunk_size=5000, overlap=500):
  chunks = []
  start = 0

  while start < len(text):
    end = min(start + chunk_size, len(text))
    chunks.append(text[start:end])

    if end == len(text):
      break

    start = end - overlap

  return chunks


# Analyzing all downloaded docs
records = []

for i, document in documents_df.iterrows():
  print(
    f"Processing document {i + 1}/{len(documents_df)}: "
    f"{document['company_name']} - "
    f"{document['document_type']}"
  )

  chunks = chunk_text(document["text"])

  for chunk in chunks:
    products = extract_products(chunk)

    for product in products:
      name = product.get("new_product", "").strip()
      description = product.get(
        "product_description", ""
      ).strip()

      if not name or not description:
        continue

      # Ensuring description is under 180 characters
      description = description[:179]

      records.append({
        "company_name": document["company_name"],
        "stock_name": document["stock_name"],
        "filing_time": document["filing_time"],
        "new_product": name,
        "product_description": description
      })


# Creating the required CSV structure
columns = [
  "company_name",
  "stock_name",
  "filing_time",
  "new_product",
  "product_description"
]

results_df = pd.DataFrame(records, columns=columns)

# Let's now remove duplicate product names within each filing
results_df = results_df.drop_duplicates(
  subset=[
    "stock_name",
    "filing_time",
    "new_product"
  ]
)

# Savng results
results_df.to_csv(
  "sec_8k_products.csv",
  index=False,
  encoding="utf-8"
)

print("\nExtraction is now completed!")
print(f"Products extracted: {len(results_df)}")

display(results_df)

# Only added a sample to the CSV file due to an HTTP connection pool timeout

Processing document 1/17: NVIDIA CORP - 8-K
Processing document 2/17: NVIDIA CORP - 8-K
Processing document 3/17: NVIDIA CORP - EX-99.1


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=240)

# Final step:

In [155]:
print("Total documents:", len(documents_df))
print("Companies analyzed:", documents_df["stock_name"].nunique())
print("Products extracted:", len(results_df))

print("\nProducts by company:")
display(
  results_df.groupby("stock_name")
  .size()
  .reset_index(name="products_extracted")
)

print("\nFinal CSV preview:")
display(results_df.head(10))


Total documents: 17
Companies analyzed: 5
Products extracted: 34

Products by company:


,stock_name,products_extracted
0,NVDA,34



Final CSV preview:


,company_name,stock_name,filing_time,new_product,product_description
0,NVIDIA CORP,NVDA,2026-09-03T12:03:56.000Z,Hugging Face,"Platform and community for developing, sharing..."
1,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Announces Financial Results for Second ...,Financial results for Second Quarter Fiscal 2027
2,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera Rubin platform,Ramping into full production
3,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Spectrum™-6 switch systems,Arriving across the world’s gigascale AI facto...
4,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera,First CPU built for AI agents
5,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Groq 3 LPX,Now in full production
6,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Vera BlueField™-4 STX,New security innovations
7,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA DSX,Providing infrastructure builders a complete p...
8,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Agen,"New software, open source models and partnerships"
9,NVIDIA CORP,NVDA,2026-08-26T20:21:19.000Z,NVIDIA Blackwell,Industry-leading AI infrastructure benchmark
